In [5]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer

# 1. Setup dasar
MODEL_NAME = "indobenchmark/indobert-large-p1"
MAX_LEN = 128
BATCH_SIZE = 8

# 2. Panggil Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 3. Load data test lu (Pastikan path-nya bener ya!)
test_df = pd.read_csv("../data_labelling/test_labeled.csv")
test_df["label"] = test_df["label"].astype(int)

# 4. Bikin Class Dataset (Kayak di code awal lu)
class SentimentDataset(Dataset):
    def __init__(self, dataframe):
        self.texts = dataframe["cleaned_text"].values
        self.labels = dataframe["label"].values
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# 5. Bungkus jadi test_loader
test_dataset = SentimentDataset(test_df)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"Beres! test_loader udah siap bawa {len(test_dataset)} data ujian.")

Beres! test_loader udah siap bawa 250 data ujian.


In [6]:
import torch
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import classification_report

print("=== SIDANG PEMBUKTIAN FILE .PT ===")

# 1. Definisi Device (INI YANG KETINGGALAN)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device yang dipake: {device}")

FILE_YANG_MAU_DICEK = 'best_indobert_model.pt' 
MODEL_NAME = "indobenchmark/indobert-large-p1"
NUM_LABELS = 3

print(f"Mengecek ingatan model dari: {FILE_YANG_MAU_DICEK}")

# Bikin wadah model kosong (Warning merah di sini cuekin aja)
model_detektif = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

# Masukin otak (.pt) ke dalam wadah model
try:
    model_detektif.load_state_dict(torch.load(FILE_YANG_MAU_DICEK, map_location=device))
    model_detektif.to(device)
    model_detektif.eval()
    
    test_preds = []
    test_labels = []

    # ASUMSI: test_loader udah lu run/define di cell sebelumnya ya!
    with torch.no_grad():
        for batch in test_loader: 
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            outputs = model_detektif(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            
            preds = torch.argmax(outputs.logits, dim=1)
            test_preds.extend(preds.cpu().numpy())
            test_labels.extend(labels.cpu().numpy())

    print("\n=== HASIL UJIANNYA ===")
    print(classification_report(test_labels, test_preds))
    
except Exception as e:
    print(f"Gagal ngeload file! Error: {e}")

=== SIDANG PEMBUKTIAN FILE .PT ===
Device yang dipake: cuda
Mengecek ingatan model dari: best_indobert_model.pt


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-large-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== HASIL UJIANNYA ===
              precision    recall  f1-score   support

           0       0.83      0.77      0.80       130
           1       0.72      0.57      0.64        60
           2       0.63      0.87      0.73        60

    accuracy                           0.74       250
   macro avg       0.73      0.73      0.72       250
weighted avg       0.76      0.74      0.74       250

